# FlowGuard Label-Histogram Defense Experiments

This notebook evaluates **FlowGuardLabelHistogramDefense (C4)** as a post-hoc/output-side detector on the saved attack-suite artifacts. Unlike C1/C3, C4 is not currently part of `evaluate_flowguard_suite.py`, so this notebook reads victim soft-labels from transfer-set artifacts and analyzes top-1 label distributions.

## What This Defense Tests

C4 is not an input anomaly detector. It asks whether the labels returned by the victim look like ordinary user traffic or like model-stealing coverage of the class space.

Expected behavior:
- `a1_clean_transfer` and `a5_labelmaze` can bypass FlowPure visually because their inputs are clean.
- Their label distribution can still be suspicious if the attacker samples for extraction utility rather than normal user intent.
- Generator attacks such as MAZE may be visible both from input geometry and output-label behavior.
- This notebook uses MMD between top-1 label histograms as the C4 score.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "notebooks" / "flow_matching_defenses"))

from flowguard_notebook_utils import (
    ATTACKS,
    apply_nature_style,
    defense_rows,
    display_defense_intro,
    display_metric_table,
    plot_defense_metrics,
    plot_label_histogram_analysis,
    show_attack_query_grid,
)

apply_nature_style()
DEFENSE = "flowguard_labelhist"
SOURCE_DEFENSE = "flowpure"
RUN_ROOT = project_root / "runs" / "notebook" / "flowguard_suite"
print(f"Project root: {project_root}")
print(f"Suite root: {RUN_ROOT}")

In [ ]:
display_defense_intro(DEFENSE)

## Evaluation Note

`evaluate_flowguard_suite.py` currently supports `flowpure`, `flowguard_integral`, and `flowguard_userlevel`. C4 needs victim outputs in `after_query`, plus a benign label histogram. The saved transfer sets contain soft labels from the victim, so this notebook evaluates the C4 idea post-hoc without re-running the full attack grid.

In [ ]:
print("Input-geometry baseline for context: FlowPure rows used as source artifacts")
flowpure_rows = defense_rows(RUN_ROOT, SOURCE_DEFENSE)
display_metric_table(flowpure_rows)
plot_defense_metrics(flowpure_rows, SOURCE_DEFENSE)

## C4 Label-Histogram Signal

The first plot is the C4 score: RBF-MMD between each attack's top-1 victim-label histogram and a benign reference histogram. The second plot compares each attack histogram directly against the benign histogram.

In [ ]:
plot_label_histogram_analysis(RUN_ROOT, source_defense=SOURCE_DEFENSE)

## Attack Query Visualization

These images are loaded from the same attack-suite artifacts used for the label histograms. The key point for C4 is that even visually clean queries can leak an extraction pattern through the labels they request from the victim.

In [ ]:
for attack in ATTACKS:
    show_attack_query_grid(RUN_ROOT, SOURCE_DEFENSE, attack, count=6)

## Reading This Notebook

If an attack has low FlowPure detection but a high label-histogram MMD, it is exactly the kind of case C4 is meant to cover: visually/geometry-clean stealing behavior. If both FlowPure detection and label MMD are low, the attacker is close to benign on both the input and output side, so you need either longer user windows, richer behavior features, or a stronger active-learning-specific detector.